# Volume 1, Chapter 8: Continued Fractions and Pell's Equation (Python companion)

Exercises first, solutions at the end. Computer-science threads: **best rational approximation** (what
`Fraction.limit_denominator` does), an exact **Pell solver**, and **Wiener's attack** on RSA keys with a small
private exponent, which is nothing but Legendre's theorem (Theorem 8.4).

In [ ]:
import math
import random
from fractions import Fraction
from decimal import Decimal, getcontext
from sympy import isprime, randprime, nextprime, E, pi, N as numeric
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10108)
seed_sympy(20108)


## Before the experiment: The next quotient is a spoiler

Guess why $355/113$ is excellent before printing the next partial quotient of $\pi$. A huge next quotient means the current convergent is unusually accurate. Decimal input is a rational approximation: keep track of how many digits you actually supplied.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P8.1 (★) Expansions.** Write `cf(x, n)` returning the first `n` partial quotients of a `Fraction` (or of an exact real given by enough digits), and `convergents(a)`. Check $\frac{415}{93}=[4;2,6,7]$, then look at $\pi$ (why is $\frac{355}{113}$ so good?) and at $e$: guess the pattern.

In [ ]:
# Your code here


**P8.2 (★★) CS application: best approximation with a bounded denominator.** Given $x$ and $Q$, find the fraction $\frac pq$ with $q\le Q$ closest to $x$. Compare with `Fraction(x).limit_denominator(Q)`, and use it to recover a screen ratio from $1.7777$ and the best approximation of $\sqrt2$ with denominator at most $100$.

*How to think about it:* by Lemma 8.3 the answer is a convergent or a *semiconvergent* $\frac{p_{n-1}+tp_n}{q_{n-1}+tq_n}$ ($0<t<a_{n+1}$) built from the last convergent with $q_n\le Q$; only two candidates need to be compared.

In [ ]:
# Your code here


**P8.3 (★★) An exact Pell solver.** For $\sqrt d$ the complete quotients are $\frac{m_k+\sqrt d}{c_k}$ with integers $m_k,c_k$: $m_{k+1}=a_kc_k-m_k$, $c_{k+1}=\frac{d-m_{k+1}^2}{c_k}$, $a_{k+1}=\left\lfloor\frac{a_0+m_{k+1}}{c_{k+1}}\right\rfloor$. Implement it, find the period of $\sqrt d$, and the first convergent $\frac pq$ with $p^2-dq^2=1$. Solve $d=61$ and $d=109$, and tabulate the fundamental solutions for $d\le100$. Which $d$ has the largest one?

In [ ]:
# Your code here


**P8.4 (★★★) CS application: Wiener's attack (1990).** Let $N=pq$ be an RSA modulus with $q<p<2q$, and $ed\equiv1\pmod{\varphi(N)}$. If $d<\frac13N^{1/4}$, then $\frac kd$ (where $ed-k\varphi(N)=1$) satisfies $\left|\frac eN-\frac kd\right|<\frac1{2d^2}$, so it is a convergent of $\frac eN$ by Legendre's theorem. Build such a weak key with $N$ of about 1024 bits and recover $d$ (and the factorisation) from the convergents of $\frac eN$.

*How to think about it:* for each convergent $\frac kd$, a candidate $\varphi=\frac{ed-1}k$ must be an integer, and then $p,q$ are the roots of $X^2-(N-\varphi+1)X+N$; a perfect-square discriminant confirms the guess.

In [ ]:
# Your code here


**P8.5 (★) Hurwitz, numerically.** For $x=\sqrt p$ ($p$ prime $<100$), $\pi$ and $\varphi$, compute $\min q_n^2\left|x-\frac{p_n}{q_n}\right|$ over the first $40$ convergents and compare with $\frac1{\sqrt5}\approx0.447$. Which number is closest to the bound, and why (Exercise 8.6)?

In [ ]:
# Your code here


---
## Solutions

### P8.1
We use exact `Fraction`s; for $\pi$ and $e$ we take a rational approximation with 60 correct digits, which determines the first few dozen partial quotients (each partial quotient consumes about one digit on average, by Lochs' theorem).

In [ ]:
def cf(x, n=None):
    x = Fraction(x)
    out = []
    while n is None or len(out) < n:
        a = math.floor(x)
        out.append(a)
        if x == a:
            break
        x = 1 / (x - a)
    return out

def convergents(a):
    p0, p1, q0, q1 = 1, a[0], 0, 1
    yield Fraction(p1, q1)
    for t in a[1:]:
        p0, p1 = p1, t * p1 + p0
        q0, q1 = q1, t * q1 + q0
        yield Fraction(p1, q1)

print(cf(Fraction(415, 93)), list(convergents(cf(Fraction(415, 93)))))
PI = Fraction(str(numeric(pi, 60)))
EE = Fraction(str(numeric(E, 60)))
print("pi:", cf(PI, 12), [str(c) for c in list(convergents(cf(PI, 5)))])
print("e: ", cf(EE, 25))
print("355/113 error:", float(abs(PI - Fraction(355, 113))), "  next quotient 292 makes it exceptionally good")

The pattern for $e$ is $[2;1,2,1,1,4,1,1,6,1,1,8,\dots]$ (Euler, 1737).

### P8.2

In [ ]:
def best_approx(x, Q):
    if Q < 1:
        raise ValueError("Q must be positive")
    a = cf(x)
    p0, p1, q0, q1 = 1, a[0], 0, 1           # p0/q0 = p_{n-1}/q_{n-1}, p1/q1 = p_n/q_n
    for t in a[1:]:
        if t * q1 + q0 > Q:
            s = (Q - q0) // q1                # largest semiconvergent with denominator <= Q
            cands = [Fraction(p1, q1), Fraction(s * p1 + p0, s * q1 + q0)]
            return min(cands, key=lambda f: (abs(f - Fraction(x)), f.denominator))
        p0, p1 = p1, t * p1 + p0
        q0, q1 = q1, t * q1 + q0
    return Fraction(p1, q1)

random.seed(0)
for _ in range(2000):
    x = Fraction(random.randrange(1, 10**12), random.randrange(1, 10**12))
    Q = random.randrange(1, 10**5)
    assert best_approx(x, Q) == x.limit_denominator(Q)
print("agrees with Fraction.limit_denominator on 2000 random tests")
print("1.7777 ->", best_approx(Fraction('1.7777'), 20))
SQ2 = Fraction(math.isqrt(2 * 10**60), 10**30)
print("sqrt2, denominator <= 100 ->", best_approx(SQ2, 100))

### P8.3
Why the recurrence stays in the integers: one checks by induction that $c_k\mid d-m_k^2$, which is exactly what makes $c_{k+1}$ an integer. The period ends when $a_k=2a_0$ (a classical endpoint criterion for this square-root algorithm; the integer-state recurrence is checked directly here).

In [ ]:
def sqrt_cf(d):
    if d < 2 or math.isqrt(d)**2 == d:
        raise ValueError("d must be positive and nonsquare")
    a0 = math.isqrt(d)
    m, c, a = 0, 1, a0
    out = [a0]
    while a != 2 * a0:
        m = a * c - m
        c = (d - m * m) // c
        a = (a0 + m) // c
        out.append(a)
    return out                                # [a0; period]

def pell(d):
    a = sqrt_cf(d)
    period = a[1:]
    p0, p1, q0, q1 = 1, a[0], 0, 1
    seq = period * 2                          # one period, or two if the period is odd
    for t in [None] + seq:
        if t is not None:
            p0, p1 = p1, t * p1 + p0
            q0, q1 = q1, t * q1 + q0
        if p1 * p1 - d * q1 * q1 == 1:
            return p1, q1

for d in [2, 61, 109]:
    x, y = pell(d)
    print(d, sqrt_cf(d)[:12], (x, y))

table = {d: pell(d) for d in range(2, 101) if math.isqrt(d) ** 2 != d}
dmax = max(table, key=lambda d: table[d][0])
print("largest fundamental solution for d <= 100:", dmax, table[dmax])

### P8.4

In [ ]:
def wiener(e, N):
    for f in convergents(cf(Fraction(e, N))):
        k, d = f.numerator, f.denominator
        if k == 0 or (e * d - 1) % k:
            continue
        phi = (e * d - 1) // k
        s = N - phi + 1                       # p + q
        disc = s * s - 4 * N
        if disc >= 0 and math.isqrt(disc) ** 2 == disc:
            r = math.isqrt(disc)
            return d, (s + r) // 2, (s - r) // 2
    return None

random.seed(2)
while True:
    q = randprime(2**511, 2**512)
    p = nextprime(q + random.randrange(2**509, 2**510))
    if q < p < 2 * q:
        break
N, phi = p * q, (p - 1) * (q - 1)
bound = math.isqrt(math.isqrt(N)) // 3
while True:
    d = random.randrange(bound // 2, bound)
    if math.gcd(d, phi) == 1:
        break
e = pow(d, -1, phi)
found = wiener(e, N)
print("N has", N.bit_length(), "bits; d has", d.bit_length(), "bits")
print("recovered d:", found[0] == d, "; factors:", {found[1], found[2]} == {p, q})

### P8.5

In [ ]:
def hurwitz_min(x, n=40):
    a = cf(x, n + 1)
    return min(c.denominator ** 2 * abs(Fraction(x) - c) for c in list(convergents(a))[2:n])

def exact(v, digits=120):
    return Fraction(str(numeric(v, digits)))

from sympy import sqrt as ssqrt, GoldenRatio, primerange
vals = {f"sqrt{p}": exact(ssqrt(p)) for p in primerange(2, 100)}
vals["pi"], vals["phi"] = exact(pi), exact(GoldenRatio)
res = sorted(((float(hurwitz_min(v)), k) for k, v in vals.items()))
print("closest to 1/sqrt5 = 0.4472:", res[-3:])
print("smallest:", res[:3])

The golden ratio stays closest to $\frac1{\sqrt5}$ because all its partial quotients are $1$, the smallest possible: by Theorem 8.2 the error $q_n^2|x-\frac{p_n}{q_n}|=\frac1{x_{n+1}+q_{n-1}/q_n}$ is largest when the complete quotients are small.

**Precision check.** The displayed irrational inputs are high-precision rational approximations. We inspect only the first 40 convergents; extending the calculation without increasing precision would eventually measure the error of the approximation, not the irrational number. The minimum over a finite prefix is evidence about that prefix, not a verification of an infinite theorem.


## Why the next quotient matters

Plot $q^2|\pi-p/q|$ for the first twelve convergents, using the 60-digit approximation supplied above. A small value means better approximation than the basic $1/q^2$ scale. The conspicuous dip belongs to $355/113$.


In [ ]:
import matplotlib.pyplot as plt
cvs = list(convergents(cf(PI, 12)))
quality = [float(c.denominator**2 * abs(PI-c)) for c in cvs]
plt.figure(figsize=(8, 4))
plt.semilogy(range(len(cvs)), quality, "o-", color="black")
for i, c in enumerate(cvs):
    if c == Fraction(355, 113):
        plt.annotate("355/113", (i, quality[i]), xytext=(i+1, quality[i]*2), arrowprops=dict(arrowstyle="->"))
plt.xlabel("convergent index (starting at 0)"); plt.ylabel(r"$q^2|\pi-p/q|$")
plt.title("The next partial quotient makes a visible difference")
plt.tight_layout(); plt.show()


## One more question before closing

For denominator bounds 10, 100, and 1000, which convergent or semiconvergent wins for pi? Explain every change of winner.

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
